In [ ]:
# ===============================================
# In This file:
# The Single-generator R-GAN framework is implemented on the MNIST dataset for Untargeted Attacks, according the R-GAN paper.
# This framework is also compatible with the Fashion-MNIST dataset with changing the dataset to Fashion-MNIST.
# The robustness (accuracy) for classifiers of R-net, C-net, and T-net are evaluated against attacks generated by G-net, FGSM, PGD, and C&W.
# ===============================================


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


In [ ]:

# ===============================================
# In This Cell:
# The architecture of the classifiers f-net (C-net) and R-net is defined as Model C.
# The pretrained classifier C-net described in the paper is referred to as f-net in the code (f-net in code = C-net in the paper).
# ===============================================


class ModelC(nn.Module):
    def __init__(self):
        super(ModelC, self).__init__()
        # --- Convolutional layers ---
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)  # Conv(32,3,3)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)                           # Conv(32,3,3)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)                                 # MaxPooling(2,2)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)                           # Conv(64,3,3)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)                           # Conv(64,3,3)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)                                 # MaxPooling(2,2)

        self.fc1 = nn.Linear(64 * 8 * 8, 200)                                              # FC(200)
        self.fc2 = nn.Linear(200, 10)                                                      # FC(10)

    def forward(self, x):
        # --- Block 1 ---
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool1(x)

        # --- Block 2 ---
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)

        x = x.view(x.size(0), -1)      # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = F.softmax(x, dim=1)        # Softmax for probabilities
        return x


In [ ]:
# ===============================================
# In This Cell:
# The architecture of the generator is defined.
# The generator is denoted as G in the code and as G-net in the paper (G in the code = G-net in the paper).
# ===============================================


class ResnetBlock(nn.Module):
    def __init__(self, dim, norm_layer=nn.BatchNorm2d, use_dropout=False, use_bias=False):
        super(ResnetBlock, self).__init__()
        conv_block = [
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, bias=use_bias),
            norm_layer(dim),
            nn.ReLU(True)
        ]
        if use_dropout:
            conv_block += [nn.Dropout(0.5)]
        conv_block += [
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, bias=use_bias),
            norm_layer(dim)
        ]
        self.conv_block = nn.Sequential(*conv_block)

    def forward(self, x):
        return x + self.conv_block(x)


class Generator(nn.Module):
    def __init__(self, gen_input_nc, image_nc):
        super(Generator, self).__init__()
        encoder = [
            nn.Conv2d(gen_input_nc, 8, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(8),
            nn.ReLU(),
            nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(32),
            nn.ReLU(),
        ]
        bottle_neck = [ResnetBlock(32), ResnetBlock(32), ResnetBlock(32), ResnetBlock(32)]
        decoder = [
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(16),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 8, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(8),
            nn.ReLU(),
            nn.Conv2d(8, image_nc, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        ]
        self.encoder = nn.Sequential(*encoder)
        self.bottle_neck = nn.Sequential(*bottle_neck)
        self.decoder = nn.Sequential(*decoder)

    def forward(self, x):
        return self.decoder(self.bottle_neck(self.encoder(x)))


In [ ]:

# ===============================================
# In This Cell:
# The dataset MNIST is loaded.
# For run this file with Fashion-MNIST, load Fashion-MNIST insted of MNIST.
# ===============================================


from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# === Device ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

##########################################################
# Specify the path of the downloaded MNIST dataset:
mnist_path = r"C:\Users\Administrator\Desktop\R-GAN\1- MNIST\Dataset"
###########################################################

# === Transform ===
transform = transforms.ToTensor()

# === Load MNIST from local folder ===
mnist_train = datasets.MNIST(
    root=mnist_path,
    train=True,
    transform=transform,
    download=True
)

mnist_test = datasets.MNIST(
    root=mnist_path,
    train=False,
    transform=transform,
    download=False
)

# === Dataloaders ===
batch_size = 128

train_dataloader = DataLoader(
    mnist_train, batch_size=batch_size, shuffle=True,
    num_workers=2
)

test_dataloader = DataLoader(
    mnist_test, batch_size=batch_size, shuffle=False,
    num_workers=2
)

print("Training samples:", len(mnist_train))
print("Test samples:", len(mnist_test))


Device: cuda
Training samples: 60000
Test samples: 10000


In [ ]:
# ===============================================
# In This Cell:
# The classifier f-net (C-net) is trained.
# ===============================================


f_net = ModelC().to(device)
optimizer_f = torch.optim.Adam(f_net.parameters(), lr=1e-3)

epochs_f = 40

for epoch in range(1, epochs_f+1):
    f_net.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for imgs, labels in train_dataloader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer_f.zero_grad()
        logits = f_net(imgs)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer_f.step()

        running_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"[f train] Epoch {epoch}/{epochs_f} | Loss: {running_loss/len(train_dataloader):.4f} | Acc: {acc:.2f}%")



# Freeze f_net for later use in R-GAN training
f_net.eval()
for p in f_net.parameters():
    p.requires_grad = False

[f train] Epoch 1/30 | Loss: 0.2142 | Acc: 93.07%
[f train] Epoch 2/30 | Loss: 0.0498 | Acc: 98.47%
[f train] Epoch 3/30 | Loss: 0.0342 | Acc: 98.91%
[f train] Epoch 4/30 | Loss: 0.0238 | Acc: 99.24%
[f train] Epoch 5/30 | Loss: 0.0189 | Acc: 99.39%
[f train] Epoch 6/30 | Loss: 0.0146 | Acc: 99.54%
[f train] Epoch 7/30 | Loss: 0.0142 | Acc: 99.52%
[f train] Epoch 8/30 | Loss: 0.0101 | Acc: 99.69%
[f train] Epoch 9/30 | Loss: 0.0093 | Acc: 99.67%
[f train] Epoch 10/30 | Loss: 0.0086 | Acc: 99.69%
[f train] Epoch 11/30 | Loss: 0.0087 | Acc: 99.70%
[f train] Epoch 12/30 | Loss: 0.0061 | Acc: 99.82%
[f train] Epoch 13/30 | Loss: 0.0049 | Acc: 99.83%
[f train] Epoch 14/30 | Loss: 0.0062 | Acc: 99.80%
[f train] Epoch 15/30 | Loss: 0.0041 | Acc: 99.87%
[f train] Epoch 16/30 | Loss: 0.0044 | Acc: 99.86%
[f train] Epoch 17/30 | Loss: 0.0056 | Acc: 99.84%
[f train] Epoch 18/30 | Loss: 0.0043 | Acc: 99.85%
[f train] Epoch 19/30 | Loss: 0.0039 | Acc: 99.88%
[f train] Epoch 20/30 | Loss: 0.0019 | A

In [ ]:
# ===============================================
# In This Cell:
# The R-GAN Framework is trained.
# R-net and G(G-net) are trained together competitively in the R-GAN Framework.
# The R-GAN Framework in this cell and file is implemented for the Untargeted Attack scenario.
# ===============================================


R_net = ModelC().to(device)             # R-net: The classifier that would be trained robust
G = Generator(1, 1).to(device)          # generator: input 1 channel, output 1 channel

optimizer_R = torch.optim.Adam(R_net.parameters(), lr=1e-4, weight_decay=1e-5)
optimizer_G = torch.optim.Adam(G.parameters(), lr=1e-3)

# Hyperparameters 
eps = 0.3           # Generated perturbation bounded on per-pixel (L_inf bound)
lambda_f = 5.0      # Coefficient for fooling pretrained f-net during generator G update
lambda_R = 5.0      # Coefficient for fooling R-net during generator G update
lambda_pert = 1.0   # Coefficient for perturbation L2 regularizer during generator G update
margin = 0.0        # hinge margin for adversarial hinge on logits

num_epochs = 40     
print_every = 1

# Hinge-style adv loss on logits (mean over batch)
def hinge_adv_loss_logits(logits, labels, margin=0.0):
    B = logits.shape[0]
    true_logits = logits[torch.arange(B), labels]
    other_logits = logits.clone()
    other_logits[torch.arange(B), labels] = -1e9
    max_other, _ = other_logits.max(dim=1)
    loss = torch.clamp(true_logits - max_other + margin, min=0.0)
    return loss.mean()

# Training loop
for epoch in range(1, num_epochs+1):
    G.train(); R_net.train()
    total_loss_R = 0.0
    total_loss_G = 0.0
    total_loss_f = 0.0
    total_loss_pert = 0.0
    batches = 0

    for imgs, labels in train_dataloader:
        imgs, labels = imgs.to(device), labels.to(device)
        batches += 1

        # ---------
        # 1) R-update: train R on clean + adv (G frozen)
        # ---------
        G.eval()
        with torch.no_grad():
            perturb = G(imgs)                      # in [-1,1] scaled 
            perturb = torch.clamp(perturb, -eps, eps)
            adv_imgs = torch.clamp(imgs + perturb, 0.0, 1.0)

        R_net.train()
        logits_clean = R_net(imgs)
        logits_adv   = R_net(adv_imgs)
        loss_R = F.cross_entropy(logits_clean, labels) + F.cross_entropy(logits_adv, labels)

        optimizer_R.zero_grad()
        loss_R.backward()
        optimizer_R.step()
        total_loss_R += loss_R.item()

        # ---------
        # 2) G-update: train G to fool f and R (freeze f and R params)
        # ---------
        G.train()
        R_net.eval()
        for p in R_net.parameters():
            p.requires_grad = False

        perturb = G(imgs)
        perturb = torch.clamp(perturb, -eps, eps)
        adv_imgs = torch.clamp(imgs + perturb, 0.0, 1.0)

        # loss vs pretrained f (encourage misclassification)
        logits_f = f_net(adv_imgs)
        loss_f_adv = hinge_adv_loss_logits(logits_f, labels, margin=margin)

        # loss vs R (encourage misclassification of R)
        logits_R_adv = R_net(adv_imgs)
        loss_R_adv = hinge_adv_loss_logits(logits_R_adv, labels, margin=margin)

        # perturbation regularizer 
        loss_pert = torch.mean(torch.norm(perturb.view(perturb.shape[0], -1), dim=1, p=2))

        loss_G = lambda_f * loss_f_adv + lambda_R * loss_R_adv + lambda_pert * loss_pert

        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()

        total_loss_G += loss_G.item()
        total_loss_f += loss_f_adv.item()
        total_loss_pert += loss_pert.item()

        for p in R_net.parameters():
            p.requires_grad = True

    # Epoch summaries
    avg_R = total_loss_R / batches
    avg_G = total_loss_G / batches
    avg_f = total_loss_f / batches
    avg_pert = total_loss_pert / batches
    if epoch % print_every == 0:
        print(f"[Epoch {epoch}/{num_epochs}] "
              f"Loss_R: {avg_R:.4f} | "
              f"Loss_G: {avg_G:.4f} | "
              f"Loss_f: {avg_f:.4f} | "
              f"Loss_pert(L2): {avg_pert:.4f}")

    # Evaluation on test set
    G.eval(); R_net.eval(); f_net.eval()
    correct_clean_R = correct_adv_R = 0
    correct_clean_f = correct_adv_f = 0
    total_test = 0
    with torch.no_grad():
        for imgs_t, labels_t in test_dataloader:
            imgs_t, labels_t = imgs_t.to(device), labels_t.to(device)
            total_test += labels_t.size(0)

            preds_R_clean = R_net(imgs_t).argmax(dim=1)
            preds_f_clean = f_net(imgs_t).argmax(dim=1)
            correct_clean_R += (preds_R_clean == labels_t).sum().item()
            correct_clean_f += (preds_f_clean == labels_t).sum().item()

            perturb_t = G(imgs_t)
            perturb_t = torch.clamp(perturb_t, -eps, eps)
            adv_imgs_t = torch.clamp(imgs_t + perturb_t, 0.0, 1.0)

            preds_R_adv = R_net(adv_imgs_t).argmax(dim=1)
            preds_f_adv = f_net(adv_imgs_t).argmax(dim=1)
            correct_adv_R += (preds_R_adv == labels_t).sum().item()
            correct_adv_f += (preds_f_adv == labels_t).sum().item()

    print(f" Eval - R clean acc: {100*correct_clean_R/total_test:.2f}% | R adv acc: {100*correct_adv_R/total_test:.2f}%")
    print(f"      f clean acc: {100*correct_clean_f/total_test:.2f}% | f adv acc: {100*correct_adv_f/total_test:.2f}%")




[Epoch 1/30] Loss_R: 1.7883 | Loss_G: 14.4762 | Loss_f: 0.6267 | Loss_pert(L2): 6.1252
 Eval - R clean acc: 92.96% | R adv acc: 79.99%
      f clean acc: 99.20% | f adv acc: 3.80%
[Epoch 2/30] Loss_R: 0.6866 | Loss_G: 17.2827 | Loss_f: 0.1061 | Loss_pert(L2): 5.7067
 Eval - R clean acc: 96.31% | R adv acc: 87.60%
      f clean acc: 99.20% | f adv acc: 4.79%
[Epoch 3/30] Loss_R: 0.4908 | Loss_G: 20.2797 | Loss_f: 0.0831 | Loss_pert(L2): 5.5223
 Eval - R clean acc: 97.32% | R adv acc: 89.31%
      f clean acc: 99.20% | f adv acc: 3.37%
[Epoch 4/30] Loss_R: 0.4117 | Loss_G: 21.9991 | Loss_f: 0.0699 | Loss_pert(L2): 5.4072
 Eval - R clean acc: 97.72% | R adv acc: 90.20%
      f clean acc: 99.20% | f adv acc: 3.84%
[Epoch 5/30] Loss_R: 0.3613 | Loss_G: 23.5052 | Loss_f: 0.0732 | Loss_pert(L2): 5.3359
 Eval - R clean acc: 97.89% | R adv acc: 91.70%
      f clean acc: 99.20% | f adv acc: 3.07%
[Epoch 6/30] Loss_R: 0.3171 | Loss_G: 25.1358 | Loss_f: 0.0763 | Loss_pert(L2): 5.3218
 Eval - R cle

In [ ]:

# ===============================================
# In the three cells below:
# T-net is trined using Model A.
# To train T-net with Model B or Model C, as described in the R-GAN Paper, set T-net according to the architecture of these models.
# The Architecture of Model B or C is defined in the "Architecture of Models" file.
# ===============================================


In [ ]:

train_dataloader = DataLoader(
    mnist_train, batch_size=batch_size, shuffle=True,
    num_workers=2
)

In [ ]:
# ===============================================
# In This Cell:
# The Architecture of Model A (T-net) is defined.
# ===============================================

class ModelA(nn.Module):
    def __init__(self):
        super(ModelA, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 64, kernel_size=5, stride=1, padding=2)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=5, stride=1, padding=2)

        # Dropout layers
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)

        # Fully connected layers
        self.fc1 = nn.Linear(64 * 28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Convolutional block
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.dropout1(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully connected block
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)

        # Return logits (no softmax for CrossEntropyLoss)
        return x


In [ ]:
# ===============================================
# In This Cell:
# The classifier of T-net is trained.
# ===============================================

T_net = ModelA().to(device)
optimizer_T = torch.optim.Adam(T_net.parameters(), lr=1e-3)

epochs_T = 40  

for epoch in range(1, epochs_T+1):
    T_net.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for imgs, labels in train_dataloader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer_T.zero_grad()
        logits = T_net(imgs)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer_T.step()

        running_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"[T train] Epoch {epoch}/{epochs_T} | Loss: {running_loss/len(train_dataloader):.4f} | Acc: {acc:.2f}%")



# Freeze f_net for later use in R-GAN training
T_net.eval()
for p in T_net.parameters():
    p.requires_grad = False

[T train] Epoch 1/30 | Loss: 0.1926 | Acc: 94.12%
[T train] Epoch 2/30 | Loss: 0.0705 | Acc: 97.96%
[T train] Epoch 3/30 | Loss: 0.0513 | Acc: 98.50%
[T train] Epoch 4/30 | Loss: 0.0391 | Acc: 98.81%
[T train] Epoch 5/30 | Loss: 0.0334 | Acc: 98.98%
[T train] Epoch 6/30 | Loss: 0.0275 | Acc: 99.16%
[T train] Epoch 7/30 | Loss: 0.0243 | Acc: 99.22%
[T train] Epoch 8/30 | Loss: 0.0204 | Acc: 99.34%
[T train] Epoch 9/30 | Loss: 0.0165 | Acc: 99.45%
[T train] Epoch 10/30 | Loss: 0.0168 | Acc: 99.47%
[T train] Epoch 11/30 | Loss: 0.0149 | Acc: 99.49%
[T train] Epoch 12/30 | Loss: 0.0140 | Acc: 99.52%
[T train] Epoch 13/30 | Loss: 0.0124 | Acc: 99.55%
[T train] Epoch 14/30 | Loss: 0.0129 | Acc: 99.57%
[T train] Epoch 15/30 | Loss: 0.0108 | Acc: 99.67%
[T train] Epoch 16/30 | Loss: 0.0094 | Acc: 99.71%
[T train] Epoch 17/30 | Loss: 0.0093 | Acc: 99.71%
[T train] Epoch 18/30 | Loss: 0.0094 | Acc: 99.67%
[T train] Epoch 19/30 | Loss: 0.0092 | Acc: 99.67%
[T train] Epoch 20/30 | Loss: 0.0071 | A

In [ ]:
# ================================================
# In This Cell: 
# The accuracy of T_net, R_net, and f_net for perturbed data by generator G (G-net attacker) and clean data is evaluated.
# In this file, the generator G-net produce Untargeted Attacks.
# ================================================

eps = 0.3

# Put all networks in evaluation mode
f_net.eval()
R_net.eval()
T_net.eval()
G.eval()

correct_clean_f = 0
correct_clean_R = 0
correct_clean_T = 0

correct_adv_f = 0
correct_adv_R = 0
correct_adv_T = 0

total = 0

print("Running evaluation on MNIST test set...")

with torch.no_grad():
    for imgs, labels in test_dataloader:
        imgs, labels = imgs.to(device), labels.to(device)
        total += labels.size(0)

        # ----------------------------
        # 1) CLEAN ACCURACY
        # ----------------------------
        preds_f_clean = f_net(imgs).argmax(dim=1)
        preds_R_clean = R_net(imgs).argmax(dim=1)
        preds_T_clean = T_net(imgs).argmax(dim=1)

        correct_clean_f += (preds_f_clean == labels).sum().item()
        correct_clean_R += (preds_R_clean == labels).sum().item()
        correct_clean_T += (preds_T_clean == labels).sum().item()

        # ----------------------------
        # 2) GENERATOR-PERTURBED DATA
        # ----------------------------
        perturb = G(imgs)
        perturb = torch.clamp(perturb, -eps, eps)  # L_inf bound
        adv_imgs = torch.clamp(imgs + perturb, 0.0, 1.0)

        preds_f_adv = f_net(adv_imgs).argmax(dim=1)
        preds_R_adv = R_net(adv_imgs).argmax(dim=1)
        preds_T_adv = T_net(adv_imgs).argmax(dim=1)

        correct_adv_f += (preds_f_adv == labels).sum().item()
        correct_adv_R += (preds_R_adv == labels).sum().item()
        correct_adv_T += (preds_T_adv == labels).sum().item()


# =============================
# Final Accuracy Results
# =============================
clean_acc_f = 100 * correct_clean_f / total
adv_acc_f   = 100 * correct_adv_f / total

clean_acc_R = 100 * correct_clean_R / total
adv_acc_R   = 100 * correct_adv_R / total

clean_acc_T = 100 * correct_clean_T / total
adv_acc_T   = 100 * correct_adv_T / total

print("\n=========== FINAL RESULTS ===========")
print(f"f_net  | Clean: {clean_acc_f:.2f}%   | Adv (G): {adv_acc_f:.2f}%")
print(f"R_net  | Clean: {clean_acc_R:.2f}%   | Adv (G): {adv_acc_R:.2f}%")
print(f"T_net  | Clean: {clean_acc_T:.2f}%   | Adv (G): {adv_acc_T:.2f}%")
print("=====================================\n")

Running evaluation on MNIST test set...

=========== FINAL RESULTS ===========
f_net  | Clean: 99.20%   | Adv (G): 6.18%
R_net  | Clean: 99.25%   | Adv (G): 98.33%
T_net  | Clean: 98.94%   | Adv (G): 66.13%



In [ ]:
#================================================================
# In the The Following Cells:
# The accuracy of R-net and T-net against untargeted transferable attacks crafted on f-net (C-net) using FGSM, PGD, and C&W attacks is evaluated.
# ===============================================================


In [ ]:
# ===============================================
# In This Cell:
# Untargeted FGSM attacks crafted on f_net (C-net) is produced.
# The accuracy of R_net and T_net is evaluated against untargeted FGSM attacks crafted on f_net.
# ===============================================

import torch
import torch.nn.functional as F
from tqdm import tqdm

f_net.eval()
R_net.eval()
T_net.eval()

eps = 0.3   # MNIST epsilon (L_inf)

total = 0

correct_clean_f = 0
correct_clean_R = 0
correct_clean_T = 0

correct_adv_f = 0
correct_adv_R = 0
correct_adv_T = 0

# Transfer stats
f_fool_count = 0
transfer_R_on_f = 0
transfer_T_on_f = 0

print("Running FGSM crafted on f_net...")

for imgs, labels in tqdm(test_dataloader):

    imgs = imgs.to(device)
    labels = labels.to(device)
    total += labels.size(0)

    # --------------------
    # CLEAN predictions
    # --------------------
    with torch.no_grad():
        preds_f_clean = f_net(imgs).argmax(dim=1)
        preds_R_clean = R_net(imgs).argmax(dim=1)
        preds_T_clean = T_net(imgs).argmax(dim=1)

        correct_clean_f += (preds_f_clean == labels).sum().item()
        correct_clean_R += (preds_R_clean == labels).sum().item()
        correct_clean_T += (preds_T_clean == labels).sum().item()

    # --------------------
    # FGSM attack on f_net
    # --------------------
    imgs_adv = imgs.clone().detach().requires_grad_(True)
    logits = f_net(imgs_adv)
    loss = F.cross_entropy(logits, labels)
    loss.backward()

    # FGSM step (L_inf)
    fgsm = eps * imgs_adv.grad.sign()
    adv_imgs = torch.clamp(imgs_adv + fgsm, 0.0, 1.0).detach()

    # --------------------
    # Adv predictions
    # --------------------
    with torch.no_grad():
        preds_f_adv = f_net(adv_imgs).argmax(dim=1)
        preds_R_adv = R_net(adv_imgs).argmax(dim=1)
        preds_T_adv = T_net(adv_imgs).argmax(dim=1)

        correct_adv_f += (preds_f_adv == labels).sum().item()
        correct_adv_R += (preds_R_adv == labels).sum().item()
        correct_adv_T += (preds_T_adv == labels).sum().item()

        # Track only samples fooled by f_net
        f_fool_mask = (preds_f_adv != labels)
        num_f_fool = f_fool_mask.sum().item()
        f_fool_count += num_f_fool

        if num_f_fool > 0:
            transfer_R_on_f += ((preds_R_adv != labels) & f_fool_mask).sum().item()
            transfer_T_on_f += ((preds_T_adv != labels) & f_fool_mask).sum().item()


# ==========================
# Compute Final Accuracies
# ==========================

clean_acc_f = 100 * correct_clean_f / total
clean_acc_R = 100 * correct_clean_R / total
clean_acc_T = 100 * correct_clean_T / total

adv_acc_f = 100 * correct_adv_f / total
adv_acc_R = 100 * correct_adv_R / total
adv_acc_T = 100 * correct_adv_T / total

transfer_rate_R = 100 * transfer_R_on_f / f_fool_count if f_fool_count > 0 else 0
transfer_rate_T = 100 * transfer_T_on_f / f_fool_count if f_fool_count > 0 else 0


# ==========================
# PRINT RESULTS
# ==========================

print("\n=== FGSM (crafted on f_net) transfer evaluation ===")
print(f"Total test samples: {total}")

print("\nClean accuracy:")
print(f"  f_net: {clean_acc_f:.2f}%")
print(f"  R_net: {clean_acc_R:.2f}%")
print(f"  T_net: {clean_acc_T:.2f}%")

print(f"\nAdv (FGSM eps={eps:.3f}) accuracy (models evaluated on adv crafted for f_net):")
print(f"  f_net: {adv_acc_f:.2f}% (drop {clean_acc_f-adv_acc_f:.2f} pp)")
print(f"  R_net: {adv_acc_R:.2f}% (drop {clean_acc_R-adv_acc_R:.2f} pp)")
print(f"  T_net: {adv_acc_T:.2f}% (drop {clean_acc_T-adv_acc_T:.2f} pp)")

print(f"\nFGSM attack success on f_net (f_net fooled count): {f_fool_count} / {total} "
      f"({100*f_fool_count/total:.2f}%)")

print(f"Of those f_net-fooled examples, transfer rate -> R_net: {transfer_rate_R:.2f}%  "
      f"|  T_net: {transfer_rate_T:.2f}%")


Running FGSM crafted on f_net...


100%|██████████████████████████████████████████████████████████████████████████████████| 79/79 [00:06<00:00, 11.31it/s]


=== FGSM (crafted on f_net) transfer evaluation ===
Total test samples: 10000

Clean accuracy:
  f_net: 99.20%
  R_net: 99.25%
  T_net: 99.38%

Adv (FGSM eps=0.300) accuracy (models evaluated on adv crafted for f_net):
  f_net: 48.64% (drop 50.56 pp)
  R_net: 97.69% (drop 1.56 pp)
  T_net: 58.42% (drop 40.96 pp)

FGSM attack success on f_net (f_net fooled count): 5136 / 10000 (51.36%)
Of those f_net-fooled examples, transfer rate -> R_net: 4.44%  |  T_net: 50.62%


In [ ]:

# ===============================================
# In This Cell:
# Untargeted PGD attacks crafted on f_net (C-net) is produced.
# The accuracy of R_net and T_net is evaluated against untargeted PGD attacks crafted on f_net.
# ===============================================

import torch
import torch.nn.functional as F
from tqdm import tqdm

f_net.eval()
R_net.eval()
T_net.eval()

# PGD hyperparameters for MNIST
eps = 0.3             # L_inf bound
pgd_steps = 30         
step_size = 0.01       # alpha

total = 0

correct_clean_f = 0
correct_clean_R = 0
correct_clean_T = 0

correct_adv_f = 0
correct_adv_R = 0
correct_adv_T = 0

# Transferability counters
f_fool_count = 0
transfer_R_on_f = 0
transfer_T_on_f = 0

print(f"Running PGD attack crafted on f_net (eps={eps}, steps={pgd_steps}, alpha={step_size})...")

for imgs, labels in tqdm(test_dataloader):
    imgs = imgs.to(device)
    labels = labels.to(device)
    total += labels.size(0)

    # ----------------------------
    # CLEAN predictions
    # ----------------------------
    with torch.no_grad():
        preds_f_clean = f_net(imgs).argmax(dim=1)
        preds_R_clean = R_net(imgs).argmax(dim=1)
        preds_T_clean = T_net(imgs).argmax(dim=1)

        correct_clean_f += (preds_f_clean == labels).sum().item()
        correct_clean_R += (preds_R_clean == labels).sum().item()
        correct_clean_T += (preds_T_clean == labels).sum().item()

    # ----------------------------
    # PGD ATTACK ON f_net
    # ----------------------------
    adv = imgs.clone().detach()

    for _ in range(pgd_steps):
        adv.requires_grad_(True)
        logits = f_net(adv)
        loss = F.cross_entropy(logits, labels)
        loss.backward()

        with torch.no_grad():
            adv = adv + step_size * adv.grad.sign()
            adv = torch.max(torch.min(adv, imgs + eps), imgs - eps)  # projection to L_inf ball
            adv = torch.clamp(adv, 0.0, 1.0)

    adv_imgs = adv.detach()

    # ----------------------------
    # Adv predictions
    # ----------------------------
    with torch.no_grad():
        preds_f_adv = f_net(adv_imgs).argmax(dim=1)
        preds_R_adv = R_net(adv_imgs).argmax(dim=1)
        preds_T_adv = T_net(adv_imgs).argmax(dim=1)

        correct_adv_f += (preds_f_adv == labels).sum().item()
        correct_adv_R += (preds_R_adv == labels).sum().item()
        correct_adv_T += (preds_T_adv == labels).sum().item()

        # Focus only on samples where f_net is fooled
        f_fool_mask = (preds_f_adv != labels)
        num_f_fool = f_fool_mask.sum().item()
        f_fool_count += num_f_fool

        if num_f_fool > 0:
            transfer_R_on_f += ((preds_R_adv != labels) & f_fool_mask).sum().item()
            transfer_T_on_f += ((preds_T_adv != labels) & f_fool_mask).sum().item()


# ======================
# Compute metrics
# ======================

clean_acc_f = 100 * correct_clean_f / total
clean_acc_R = 100 * correct_clean_R / total
clean_acc_T = 100 * correct_clean_T / total

adv_acc_f = 100 * correct_adv_f / total
adv_acc_R = 100 * correct_adv_R / total
adv_acc_T = 100 * correct_adv_T / total

transfer_rate_R = 100 * transfer_R_on_f / f_fool_count if f_fool_count > 0 else 0
transfer_rate_T = 100 * transfer_T_on_f / f_fool_count if f_fool_count > 0 else 0


# ======================
# PRINT RESULTS
# ======================

print("\n=== PGD (crafted on f_net) transfer evaluation ===")
print(f"Total test samples: {total}")

print("\nClean accuracy:")
print(f"  f_net: {clean_acc_f:.2f}%")
print(f"  R_net: {clean_acc_R:.2f}%")
print(f"  T_net: {clean_acc_T:.2f}%")

print(f"\nAdv (PGD eps={eps:.3f}, steps={pgd_steps}) accuracy (models evaluated on adv crafted for f_net):")
print(f"  f_net: {adv_acc_f:.2f}% (drop {clean_acc_f-adv_acc_f:.2f} pp)")
print(f"  R_net: {adv_acc_R:.2f}% (drop {clean_acc_R-adv_acc_R:.2f} pp)")
print(f"  T_net: {adv_acc_T:.2f}% (drop {clean_acc_T-adv_acc_T:.2f} pp)")

print(f"\nPGD fooled count on f_net: {f_fool_count} / {total} ({100*f_fool_count/total:.2f}%)")
print(f"Of those f_net-fooled examples, transfer rate -> R_net: {transfer_rate_R:.2f}%  |  T_net: {transfer_rate_T:.2f}%")


Running PGD attack crafted on f_net (eps=0.3, steps=30, alpha=0.01)...


100%|██████████████████████████████████████████████████████████████████████████████████| 79/79 [00:14<00:00,  5.45it/s]


=== PGD (crafted on f_net) transfer evaluation ===
Total test samples: 10000

Clean accuracy:
  f_net: 99.20%
  R_net: 99.25%
  T_net: 99.38%

Adv (PGD eps=0.300, steps=30) accuracy (models evaluated on adv crafted for f_net):
  f_net: 3.34% (drop 95.86 pp)
  R_net: 98.14% (drop 1.11 pp)
  T_net: 54.77% (drop 44.61 pp)

PGD fooled count on f_net: 9666 / 10000 (96.66%)
Of those f_net-fooled examples, transfer rate -> R_net: 1.92%  |  T_net: 46.66%


In [ ]:

# ===============================================
# In This Cell:
# Untargeted C&W attacks crafted on f_net (C-net) is produced.
# The accuracy of R_net and T_net is evaluated against untargeted C&W attacks crafted on f_net.
# ===============================================


import torch
import torch.nn.functional as F
from tqdm import tqdm

f_net.eval()
R_net.eval()
T_net.eval()

# ----------------------------
# C&W hyperparameters
# ----------------------------
cw_steps = 1000          # optimization steps
lr = 0.01               # Adam learning rate
c = 0.1                # weight on L2 term
kappa = 0.0             # confidence margin
early_stop = True

total = 0

correct_clean_f = 0
correct_clean_R = 0
correct_clean_T = 0

correct_adv_f = 0
correct_adv_R = 0
correct_adv_T = 0

# Transfer statistics
f_fool_count = 0
transfer_R = 0
transfer_T = 0

criterion = torch.nn.CrossEntropyLoss(reduction="none")

print(f"Running C&W attack (f_net white-box): steps={cw_steps}, lr={lr}, c={c}, kappa={kappa}")

for imgs, labels in tqdm(test_dataloader):

    imgs = imgs.to(device)
    labels = labels.to(device)
    B = imgs.size(0)
    total += B

    # ----------------------------
    # CLEAN predictions
    # ----------------------------
    with torch.no_grad():
        preds_f_clean = f_net(imgs).argmax(dim=1)
        preds_R_clean = R_net(imgs).argmax(dim=1)
        preds_T_clean = T_net(imgs).argmax(dim=1)

        correct_clean_f += (preds_f_clean == labels).sum().item()
        correct_clean_R += (preds_R_clean == labels).sum().item()
        correct_clean_T += (preds_T_clean == labels).sum().item()

    # ----------------------------
    # C&W ATTACK on f_net
    # Work in pixel space [0,1]
    # ----------------------------
    delta = torch.zeros_like(imgs, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)

    for step in range(cw_steps):
        opt.zero_grad()

        adv = torch.clamp(imgs + delta, 0.0, 1.0)
        logits = f_net(adv)

        # hinge loss: max(true - max_others + kappa, 0)
        true_logits = logits[torch.arange(B), labels]
        other_logits = logits.clone()
        other_logits[torch.arange(B), labels] = -1e9
        max_other, _ = other_logits.max(dim=1)

        margin = true_logits - max_other + kappa
        hinge = torch.clamp(margin, min=0.0).mean()

        # L2 penalty
        l2 = (delta.view(B, -1).pow(2).sum(dim=1)).mean()

        loss = hinge + c * l2
        loss.backward()
        opt.step()

        # Optional early stop
        if early_stop:
            with torch.no_grad():
                preds_now = f_net(adv).argmax(dim=1)
                if (preds_now != labels).all():
                    break

    # Final adversarial images
    with torch.no_grad():
        adv = torch.clamp(imgs + delta, 0.0, 1.0)

        preds_f_adv = f_net(adv).argmax(dim=1)
        preds_R_adv = R_net(adv).argmax(dim=1)
        preds_T_adv = T_net(adv).argmax(dim=1)

        correct_adv_f += (preds_f_adv == labels).sum().item()
        correct_adv_R += (preds_R_adv == labels).sum().item()
        correct_adv_T += (preds_T_adv == labels).sum().item()

        # Transfer statistics: only samples fooled by f_net
        f_mask = (preds_f_adv != labels)
        num_fool = f_mask.sum().item()
        f_fool_count += num_fool

        if num_fool > 0:
            transfer_R += ((preds_R_adv != labels) & f_mask).sum().item()
            transfer_T += ((preds_T_adv != labels) & f_mask).sum().item()


# ============================================================
# FINAL METRICS
# ============================================================

clean_acc_f = 100 * correct_clean_f / total
clean_acc_R = 100 * correct_clean_R / total
clean_acc_T = 100 * correct_clean_T / total

adv_acc_f = 100 * correct_adv_f / total
adv_acc_R = 100 * correct_adv_R / total
adv_acc_T = 100 * correct_adv_T / total

transfer_R_rate = 100 * transfer_R / f_fool_count if f_fool_count > 0 else 0
transfer_T_rate = 100 * transfer_T / f_fool_count if f_fool_count > 0 else 0

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n=== C&W L2 (crafted on f_net) transfer evaluation ===")
print(f"Total test samples: {total}")

print("\nClean accuracy:")
print(f"  f_net: {clean_acc_f:.2f}%")
print(f"  R_net: {clean_acc_R:.2f}%")
print(f"  T_net: {clean_acc_T:.2f}%")

print(f"\nAdv (C&W eps unconstrained, steps={cw_steps}) accuracy (crafted for f_net):")
print(f"  f_net: {adv_acc_f:.2f}% (drop {clean_acc_f-adv_acc_f:.2f} pp)")
print(f"  R_net: {adv_acc_R:.2f}% (drop {clean_acc_R-adv_acc_R:.2f} pp)")
print(f"  T_net: {adv_acc_T:.2f}% (drop {clean_acc_T-adv_acc_T:.2f} pp)")

print(f"\nC&W attack success on f_net: {f_fool_count} / {total} "
      f"({100*f_fool_count/total:.2f}%)")

print(f"Of those f_net-fooled examples, transfer rate → R_net: {transfer_R_rate:.2f}%")
print(f"Of those f_net-fooled examples, transfer rate → T_net: {transfer_T_rate:.2f}%")


Running C&W attack (f_net white-box): steps=1000, lr=0.01, c=0.1, kappa=0.0


100%|██████████████████████████████████████████████████████████████████████████████████| 79/79 [00:29<00:00,  2.70it/s]


=== C&W L2 (crafted on f_net) transfer evaluation ===
Total test samples: 10000

Clean accuracy:
  f_net: 99.20%
  R_net: 99.25%
  T_net: 99.38%

Adv (C&W eps unconstrained, steps=1000) accuracy (crafted for f_net):
  f_net: 0.00% (drop 99.20 pp)
  R_net: 99.01% (drop 0.24 pp)
  T_net: 79.66% (drop 19.72 pp)

C&W attack success on f_net: 10000 / 10000 (100.00%)
Of those f_net-fooled examples, transfer rate → R_net: 0.99%
Of those f_net-fooled examples, transfer rate → T_net: 20.34%
